# ทำนายกิจกรรมแบบเรียลไทม์จาก phyphox

รับค่า accelerometer และ gyroscope จากสมาร์ทโฟนผ่าน Wi-Fi แล้วทำนายกิจกรรมด้วยโมเดลที่ 2 (`model2.npz`)
ผลลัพธ์แสดงทุก 1 วินาที โดยสรุปจากผลการทำนายในช่วง 5 วินาทีล่าสุด

**วิธีใช้งาน**
1. วางไฟล์ `model2.npz` ไว้ในโฟลเดอร์เดียวกับ notebook นี้
2. ใส่ค่า `SCALE_MIN` และ `SCALE_MAX` ในเซลล์ถัดไป (หากใน `model2.npz` ยังไม่มีค่าดังกล่าว)
3. เปิดการทดลองใน phyphox แล้วเปิด Allow remote access จากนั้นแก้ `URL` ให้ตรงกับที่แอปแสดง
4. กด Run All และหยุดการทำงานด้วยปุ่ม Interrupt

In [6]:
# ===== ค่าที่ต้องตั้ง =====
URL        = "http://10.20.122.173:8080"   # URL ที่ phyphox แสดง
MODEL_PATH = "model2.npz"

# ค่า min และ max ที่ใช้ปรับสเกลฟีเจอร์ทั้ง 24 ตัว เรียงตาม feature_cols ใน model2.npz
# ใช้เมื่อใน model2.npz ยังไม่มีค่า scale_min / scale_max
SCALE_MIN = None   # เช่น [ -0.5, 0.01, ... ]  (24 ค่า)
SCALE_MAX = None   # เช่น [  0.8, 2.30, ... ]  (24 ค่า)

STD_DDOF   = 1      # ส่วนเบี่ยงเบนมาตรฐาน: หารด้วย n − 1 ตรงกับตอนสร้าง features_ready_2.csv
# =========================

FS         = 50     # อัตราการเก็บค่า (ค่าต่อวินาที)
WIN        = 128    # ความยาวช่วงที่ใช้ทำนาย 1 ครั้ง (128 ค่า = 2.56 วินาที)
PRED_EVERY = 0.5    # ทำนายทุก 0.5 วินาที
VOTE_SEC   = 5.0    # สรุปผลจากช่วง 5 วินาทีล่าสุด
SHOW_EVERY = 1.0    # แสดงผลทุก 1 วินาที

ACC = ["accX", "accY", "accZ"]
GYR = ["gyrX", "gyrY", "gyrZ"]
THAI = {"walking": "เดิน", "upstairs": "ขึ้นบันได", "downstairs": "ลงบันได",
        "sitting": "นั่ง", "standing": "ยืน", "laying_left": "นอน"}

## 1. โหลดโมเดล

โมเดลประกอบด้วย μ, W, centroid และชื่อกิจกรรม พร้อมค่า min / max สำหรับปรับสเกลฟีเจอร์

In [7]:
import json, time, urllib.request
import numpy as np
from IPython.display import clear_output

m = np.load(MODEL_PATH, allow_pickle=False)
FEATURE_COLS = [str(c) for c in m["feature_cols"]]
MU, W, CENTROIDS = m["mu"], m["W"], m["centroids"]
CLASSES = np.array([str(c) for c in m["classes"]])

smin = m["scale_min"] if m["scale_min"].size else (None if SCALE_MIN is None else np.asarray(SCALE_MIN, float))
smax = m["scale_max"] if m["scale_max"].size else (None if SCALE_MAX is None else np.asarray(SCALE_MAX, float))
if smin is None or smax is None:
    raise ValueError("ยังไม่มีค่า min / max สำหรับปรับสเกล: ใส่ SCALE_MIN และ SCALE_MAX ในเซลล์ด้านบน (อย่างละ 24 ค่า)")
if smin.shape != (24,) or smax.shape != (24,):
    raise ValueError(f"SCALE_MIN และ SCALE_MAX ต้องมีอย่างละ 24 ค่า (ตอนนี้ {smin.shape}, {smax.shape})")
SCALE_MIN_ARR, SCALE_MAX_ARR = smin.astype(float), smax.astype(float)

print("k =", W.shape[1], "| กิจกรรม:", [THAI.get(c, c) for c in CLASSES])

k = 7 | กิจกรรม: ['ลงบันได', 'นอน', 'นั่ง', 'ยืน', 'ขึ้นบันได', 'เดิน']


## 2. ฟังก์ชันแปลงสัญญาณและทำนาย

ข้อมูล 1 ช่วง (128 ค่า × 6 ช่องสัญญาณ) แปลงเป็นฟีเจอร์ 24 ค่า ได้แก่ mean, std, min และ max ของแต่ละช่องสัญญาณ
จากนั้นปรับสเกลให้อยู่ในช่วง −1 ถึง 1 ลบ μ ฉายด้วย W และเลือก centroid ที่ใกล้ที่สุด

In [8]:
CHANNEL_INDEX = {"ax": 0, "ay": 1, "az": 2, "gx": 3, "gy": 4, "gz": 5}
STATS = {"mean": lambda v: v.mean(), "std": lambda v: v.std(ddof=STD_DDOF),
         "min": lambda v: v.min(), "max": lambda v: v.max()}

def window_to_features(win):
    """win ขนาด (128, 6) เรียง ax, ay, az, gx, gy, gz → ฟีเจอร์ 24 ค่า เรียงตาม FEATURE_COLS"""
    feats = []
    for col in FEATURE_COLS:
        ch, st = col.split("_")
        feats.append(STATS[st](win[:, CHANNEL_INDEX[ch]]))
    return np.array(feats)

def scale(f):
    return 2 * (f - SCALE_MIN_ARR) / (SCALE_MAX_ARR - SCALE_MIN_ARR) - 1

def predict_window(win):
    z = (scale(window_to_features(win)) - MU) @ W
    d = np.linalg.norm(CENTROIDS - z, axis=1)
    return CLASSES[np.argmin(d)]

## 3. ฟังก์ชันรับข้อมูลจาก phyphox

ดึงเฉพาะค่าใหม่ที่ยังไม่เคยรับ พร้อมเวลาของแต่ละค่า แล้วปรับข้อมูล accelerometer และ gyroscope ให้อยู่บนเวลาเดียวกันที่ 50 ค่าต่อวินาที

In [9]:
def get(path):
    with urllib.request.urlopen(URL + path, timeout=3) as r:
        return json.loads(r.read())

def fetch_new(tname, names, since):
    """ค่าที่มีเวลามากกว่า since → (เวลา, ค่า 3 แกน)"""
    q = "&".join([f"{tname}={since:.6f}"] + [f"{n}={since:.6f}%7C{tname}" for n in names])
    buf = get("/get?" + q)["buffer"]
    cols = [buf[tname]["buffer"]] + [buf[n]["buffer"] for n in names]
    n = min(len(c) for c in cols)                   # บางครั้งแต่ละ buffer ส่งมาไม่เท่ากัน ใช้เท่าที่ครบทุกแกน
    arr = np.array([[np.nan if x is None else x for x in c[:n]] for c in cols], dtype=float).reshape(4, n)
    t, v = arr[0], arr[1:].T
    ok = np.isfinite(t) & np.isfinite(v).all(axis=1)
    return t[ok], v[ok]

def window_at(end, tA, A, tG, G):
    """สัญญาณ 2.56 วินาทีที่สิ้นสุด ณ เวลา end → (128, 6)"""
    grid = end - (WIN - 1) / FS + np.arange(WIN) / FS
    acc = np.column_stack([np.interp(grid, tA, A[:, i]) for i in range(3)])
    gyr = np.column_stack([np.interp(grid, tG, G[:, i]) for i in range(3)])
    return np.hstack([acc, gyr])

## 4. ทำนายแบบเรียลไทม์

- ทำนายทุก 0.5 วินาที จากสัญญาณ 2.56 วินาทีล่าสุด
- ทุก 1 วินาที นับผลการทำนายใน 5 วินาทีล่าสุด แล้วแสดงกิจกรรมที่พบมากที่สุด
- ระหว่างที่ข้อมูลยังไม่ครบ 5 วินาที จะแสดง "กำลังรวบรวมข้อมูล"

กดปุ่ม Interrupt เพื่อหยุด

In [10]:
def summarize(preds, now):
    """preds = [(เวลา, กิจกรรม)] → ข้อความสรุปจาก 5 วินาทีล่าสุด"""
    recent = [p for t, p in preds if t > now - VOTE_SEC]
    labels, counts = np.unique(recent, return_counts=True)
    order = np.argsort(-counts)
    top = labels[order[0]]
    detail = ", ".join(f"{THAI.get(labels[i], labels[i])} {counts[i]}" for i in order)
    return f"ตอนนี้: {THAI.get(top, top)}   ({detail})"

def run():
    get("/control?cmd=clear")
    get("/control?cmd=start")
    tA, A = np.empty(0), np.empty((0, 3))
    tG, G = np.empty(0), np.empty((0, 3))
    preds, next_pred, next_show = [], None, None
    shown = False
    try:
        while True:
            t, v = fetch_new("acc_time", ACC, tA[-1] if len(tA) else -1)
            tA, A = np.append(tA, t), np.vstack([A, v])
            t, v = fetch_new("gyr_time", GYR, tG[-1] if len(tG) else -1)
            tG, G = np.append(tG, t), np.vstack([G, v])

            if len(tA) and len(tG):
                start = max(tA[0], tG[0]) + (WIN - 1) / FS      # เวลาแรกที่มีข้อมูลครบ 1 ช่วง
                now = min(tA[-1], tG[-1])
                if next_pred is None:
                    next_pred, next_show = start, start + VOTE_SEC
                while next_pred <= now:                          # ทำนายทุก 0.5 วินาที
                    preds.append((next_pred, predict_window(window_at(next_pred, tA, A, tG, G))))
                    next_pred += PRED_EVERY
                if now >= next_show:                             # แสดงผลทุก 1 วินาที
                    clear_output(wait=True)
                    print(f"[{now:6.1f} วินาที]  " + summarize(preds, now))
                    while next_show <= now:
                        next_show += SHOW_EVERY
                    shown = True
                elif not shown:
                    clear_output(wait=True)
                    print(f"[{now:6.1f} วินาที]  กำลังรวบรวมข้อมูล")

                keep_from = now - (VOTE_SEC + WIN / FS + 2)       # เก็บข้อมูลย้อนหลังไว้แค่พอใช้
                tA, A = tA[tA >= keep_from], A[tA >= keep_from]
                tG, G = tG[tG >= keep_from], G[tG >= keep_from]
                preds = [p for p in preds if p[0] > now - VOTE_SEC - 1]
            else:
                clear_output(wait=True)
                print("กำลังรวบรวมข้อมูล")
            time.sleep(0.2)
    except KeyboardInterrupt:
        get("/control?cmd=stop")
        print("หยุดการทำงานแล้ว")

run()

[   4.4 วินาที]  กำลังรวบรวมข้อมูล
หยุดการทำงานแล้ว
